In [1]:
import subprocess, sys, torch

tv = torch.__version__.split("+")[0]
cu = "cu" + torch.version.cuda.replace(".", "")
url = f"https://data.pyg.org/whl/torch-{tv}+{cu}.html"

subprocess.run([sys.executable,"-m","pip","install",
    "pyg_lib","torch_scatter","torch_sparse",
    "torch_cluster","torch_spline_conv","-f",url,"-q"], check=False)
subprocess.run([sys.executable,"-m","pip","install","torch_geometric","-q"])
print("✔ PyG installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 36.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 108.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 114.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 101.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 55.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 56.1 MB/s eta 0:00:00
✔ PyG installed


In [2]:
import shutil, os, sys

CODE = "/kaggle/input/datasets/jarinproma/pgcn-eeg-code"
for f in os.listdir(CODE):
    if f.endswith(".py"):
        shutil.copy(f"{CODE}/{f}", f"/kaggle/working/{f}")
        print(f"  Copied: {f}")
sys.path.insert(0, "/kaggle/working")
print("✔ Done")

  Copied: kaggle_notebooks.py
  Copied: preprocess_eeg.py
  Copied: pgcn_model.py
  Copied: pgcn_experiment.py
  Copied: preprocess_fusion.py
  Copied: pgcn_statistics_plots.py
  Copied: pgcn_train.py
  Copied: pgcn_core.py
✔ Done


In [3]:
import os, json

# Find correct path
candidates = [
    "/kaggle/input/datasets/jarinproma/pgcn-deap-fusion-preprocessed/preprocessed_fusion",
    "/kaggle/input/pgcn-deap-fusion-preprocessed/preprocessed_fusion",
    "/kaggle/input/datasets/jarinproma/pgcn-deap-fusion-preprocessed",
    "/kaggle/input/datasets/jarinproma/pgcn-deap-fusion-preprocessed/preprocessed_fusion",
]

DATA = None
for path in candidates:
    if os.path.exists(path):
        subs = [d for d in os.listdir(path) if d.startswith("s")]
        if subs:
            mp = f"{path}/{subs[0]}/meta.json"
            if os.path.exists(mp):
                m = json.load(open(mp))
                if m.get("has_real_video") is not None:
                    DATA = path
                    print(f"✔ Correct path: {DATA}")
                    print(f"  Subjects: {len(subs)}")
                    real = [s for s in subs
                            if json.load(open(f"{path}/{s}/meta.json"))
                                    .get("has_real_video", False)]
                    print(f"  Real video: {len(real)} subjects")
                    break

if DATA is None:
    print("✗ Could not find correct path")

✔ Correct path: /kaggle/input/datasets/jarinproma/pgcn-deap-fusion-preprocessed/preprocessed_fusion
  Subjects: 32
  Real video: 22 subjects


In [4]:
from pgcn_experiment import run_experiment, print_summary

# Session 1: folds s01-s07 (~5-6h)
store = run_experiment(
    data_dir = DATA,
    out_dir  = "/kaggle/working/results_fusion/loso_early",
    mode     = "loso",
    fusion   = "early",
    ablation = "none",
    force    = False,
)
print_summary(store,
    out_dir="/kaggle/working/results_fusion/loso_early/aggregate")
print("✔ Early Fusion LOSO done!")

  mode=loso  fusion=early  ablation=none
  data_dir : /kaggle/input/datasets/jarinproma/pgcn-deap-fusion-preprocessed/preprocessed_fusion
  out_dir  : /kaggle/working/results_fusion/loso_early

  loso  fusion=early  ablation=none  (22 test folds)

───────────────────────────────────────────────────────
  fold test=s01  train=31
  DEAPDataset [train]   25172 windows  (31 subjects)
  DEAPDataset [val  ]    5394 windows  (31 subjects)
  DEAPDataset [all  ]    1160 windows  (1 subjects)
  ep  1/100  tr=6.179  val=0.706  aro=0.729  mean=0.723
  ep 10/100  tr=4.659  val=0.755  aro=0.752  mean=0.761
  ep 20/100  tr=4.483  val=0.780  aro=0.707  mean=0.755
  Early stop ep22  best=0.7663
  TEST  val=0.0489  aro=0.5275  dom=0.1045  lik=0.9647  quad=0.2207

───────────────────────────────────────────────────────
  fold test=s02  train=31
  DEAPDataset [train]   25172 windows  (31 subjects)
  DEAPDataset [val  ]    5394 windows  (31 subjects)
  DEAPDataset [all  ]    1160 windows  (1 subjects)
  ep

KeyboardInterrupt: 

In [5]:
import shutil, os

shutil.make_archive("/kaggle/working/loso_early_results", "zip",
                    "/kaggle/working", "results_fusion")
print(f"✔ {os.path.getsize('/kaggle/working/loso_early_results.zip')/1e6:.1f} MB")

# Check completed folds
OUT = "/kaggle/working/results_fusion/loso_early"
done = []
if os.path.exists(OUT):
    for fd in sorted(os.scandir(OUT), key=lambda x: x.name):
        if fd.name.startswith("fold_") and \
           os.path.exists(f"{fd.path}/metrics.json"):
            done.append(fd.name.replace("fold_",""))
print(f"Completed: {len(done)}/22 folds → {done}")
remaining = [f"s{i:02d}" for i in range(1,23) if f"s{i:02d}" not in done]
print(f"Remaining: {remaining}")

✔ 29.3 MB
Completed: 9/22 folds → ['s01', 's02', 's03', 's04', 's05', 's06', 's07', 's08', 's09']
Remaining: ['s10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21', 's22']
